In [ ]:
import os
import polars as pl
from time import time
from pathlib import Path

FILE_TYPES = [
    "CAREG",
    "EHR_DIAGNOSES",
    "GENOMIC_SPECIMEN",
    "CNV",
    "SNV",
    "SV",
    "HEALTH_HISTORY",
    "MEDICATIONS",
    "MEDICATIONS_SUMMARY",
    "LABS",
    "PT_INFO_STATUS_REGISTRATION",
    "TREATMENT_PLAN",
]

SOMATIC_RESULT_FILE_TYPES = {"SNV", "CNV", "SV"}
SOMATIC_GROUP_KEY = ["DFCI_MRN", "SAMPLE_ACCESSION_NBR", "TEST_TYPE"]

FILE_MAP = {
    "ALL_2021_11": {
        "CAREG": "CANCER_DIAGNOSIS_CAREG.csv.gz",
        "EHR_DIAGNOSES": "EHR_DIAGNOSIS.csv.gz",
        "GENOMIC_SPECIMEN": "GENOMIC_SPECIMEN.csv.gz",
        "CNV": "GENOMIC_CNV_RESULTS.csv.gz",
        "SNV": "GENOMIC_MUTATION_RESULTS.csv.gz",
        "SV": "GENOMIC_SV_RESULTS.csv.gz",
        "HEALTH_HISTORY": "HEALTH_HISTORY.csv.gz",
        "MEDICATIONS": "MEDICATIONS.csv.gz",
        "MEDICATIONS_SUMMARY": "MEDICATIONS_SUMMARY.csv.gz",
        "LABS": "OUTPT_LAB_RESULTS_LABS.csv.gz",
        "PT_INFO_STATUS_REGISTRATION": "PT_INFO_STATUS_REGISTRATION.csv.gz",
        "TREATMENT_PLAN": "TREATMENT_PLAN.csv.gz",
    },
    "ALL_2022_11": {
        "CAREG": "CANCER_DIAGNOSIS_CAREG.csv",
        "EHR_DIAGNOSES": "EHR_DIAGNOSIS.csv",
        "GENOMIC_SPECIMEN": "GENOMIC_SPECIMEN.csv",
        "CNV": "GENOMIC_CNV_RESULTS.csv",
        "SNV": "GENOMIC_MUTATION_RESULTS.csv",
        "SV": "GENOMIC_SV_RESULTS.csv",
        "HEALTH_HISTORY": "HEALTH_HISTORY.csv",
        "MEDICATIONS": "MEDICATIONS.csv",
        "MEDICATIONS_SUMMARY": "MEDICATIONS_SUMMARY.csv",
        "LABS": "OUTPT_LAB_RESULTS_LABS.csv",
        "PT_INFO_STATUS_REGISTRATION": "PT_INFO_STATUS_REGISTRATION.csv",
        "TREATMENT_PLAN": "TREATMENT_PLAN.csv",
    },
    "ALL_2024_01": {
        "CAREG": "CANCER_DIAGNOSIS_CAREG.csv",
        "EHR_DIAGNOSES": "EHR_DIAGNOSIS.csv",
        "GENOMIC_SPECIMEN": "GENOMIC_SPECIMEN.csv",
        "CNV": "GENOMIC_CNV_RESULTS.csv",
        "SNV": "GENOMIC_MUTATION_RESULTS.csv",
        "SV": "GENOMIC_SV_RESULTS.csv",
        "HEALTH_HISTORY": "HEALTH_HISTORY.csv",
        "MEDICATIONS": "MEDICATIONS.csv",
        "MEDICATIONS_SUMMARY": "MEDICATIONS_SUMMARY.csv",
        "LABS": "OUTPT_LAB_RESULTS_LABS.csv",
        "PT_INFO_STATUS_REGISTRATION": "PT_INFO_STATUS_REGISTRATION.csv",
        "TREATMENT_PLAN": "TREATMENT_PLAN.csv",
    },
    "ALL_2024_03": {
        "CAREG": "CANCER_DIAGNOSIS_CAREG.csv",
        "EHR_DIAGNOSES": "EHR_DIAGNOSIS.csv",
        "GENOMIC_SPECIMEN": "GENOMIC_SPECIMEN.csv",
        "CNV": "GENOMIC_CNV_RESULTS.csv",
        "SNV": "GENOMIC_MUTATION_RESULTS.csv",
        "SV": "GENOMIC_SV_RESULTS.csv",
        "HEALTH_HISTORY": "HEALTH_HISTORY.csv",
        "MEDICATIONS": "MEDICATIONS.csv",
        "MEDICATIONS_SUMMARY": "MEDICATIONS_SUMMARY.csv",
        # split into _part_1 / _part_2; _merged is the concatenation
        "LABS": "OUTPT_LAB_RESULTS_LABS_merged.csv",
        "PT_INFO_STATUS_REGISTRATION": "PT_INFO_STATUS_REGISTRATION.csv",
        "TREATMENT_PLAN": "TREATMENT_PLAN.csv",
    },
    "ALL_2024_05": {
        "CAREG": "CANCER_DIAGNOSIS_CAREG.csv",
        "EHR_DIAGNOSES": "EHR_DIAGNOSIS.csv",
        "GENOMIC_SPECIMEN": "GENOMIC_SPECIMEN.csv",
        "CNV": "GENOMIC_CNV_RESULTS.csv",
        "SNV": "GENOMIC_MUTATION_RESULTS.csv",
        "SV": "GENOMIC_SV_RESULTS.csv",
        "HEALTH_HISTORY": "HEALTH_HISTORY.csv",
        "MEDICATIONS": "MEDICATIONS.csv",
        "MEDICATIONS_SUMMARY": "MEDICATIONS_SUMMARY.csv",
        "LABS": "OUTPT_LAB_RESULTS_LABS.csv",
        "PT_INFO_STATUS_REGISTRATION": "PT_INFO_STATUS_REGISTRATION.csv",
        "TREATMENT_PLAN": "TREATMENT_PLAN.csv",
    },
    "ALL_2025_03": {
        "CAREG": "CANCER_DIAGNOSIS_CAREG.csv",
        "EHR_DIAGNOSES": "EHR_DIAGNOSIS.csv",
        "GENOMIC_SPECIMEN": "SOMATIC_SPECIMEN.csv",
        "CNV": "SOMATIC_CNV_RESULTS.csv",
        "SNV": "SOMATIC_MUTATION_RESULTS.csv",
        "SV": "SOMATIC_SV_RESULTS.csv",
        "HEALTH_HISTORY": "HEALTH_HISTORY.csv",
        "MEDICATIONS": "MEDICATIONS.csv",
        "MEDICATIONS_SUMMARY": "MEDICATIONS_SUMMARY.csv",
        "LABS": "OUTPT_LAB_RESULTS_LABS.csv",
        "PT_INFO_STATUS_REGISTRATION": "PT_INFO_STATUS_REGISTRATION.csv",
        "TREATMENT_PLAN": "TREATMENT_PLAN.csv",
    },
    "ALL_2026_03": {
        "CAREG": "IDM_CARG_DIAGNOSIS.csv",
        "EHR_DIAGNOSES": "IDM_EHR_DIAGNOSIS.csv",
        "GENOMIC_SPECIMEN": "CDM_SPEC_CON.csv",
        "CNV": "CDM_CNV_CON.csv",
        "SNV": "CDM_SNV_CON.csv",
        "SV": "CDM_SV_CON.csv",
        "HEALTH_HISTORY": "IDM_OPS_HEALTH_HISTORY.csv",
        "MEDICATIONS": "IDM_OPS_MED_ORDER.csv",
        "MEDICATIONS_SUMMARY": "IDM_OPS_MED_SUMMARY.csv",
        "LABS": None,  # no lab-results table in this release
        "PT_INFO_STATUS_REGISTRATION": "IDM_D_PATIENT_STATUS.csv",  # see AMBIGUOUS
        "TREATMENT_PLAN": "IDM_OPS_CHEMO_PLAN.csv",
    },
}

COLUMN_MAP = {
    "CAREG": {
        "DFCI_MRN": pl.String,
        "DIAGNOSIS_SEQ_NBR": pl.Int16,
        "TUMOR_OCC_ORD": pl.Int16,
        "DIAGNOSIS_DT": pl.String,
        "AGE_AT_DIAGNOSIS_NBR": pl.Int16,
        "FIRST_CONTACT_DT": pl.String,
        "LAST_CONTACT_DT": pl.String,
        "SITE_CD": pl.String,
        "HISTOLOGY_CD": pl.Int64,
        "BEST_AJCC_STAGE_CD": pl.String,
        "PATH_T_CD": pl.String,
        "PATH_N_CD": pl.String,
        "PATH_M_CD": pl.String,
        "PATH_STAGE_CD": pl.String,
        "CLIN_T_CD": pl.String,
        "CLIN_N_CD": pl.String,
        "CLIN_M_CD": pl.String,
        "CLIN_STAGE_CD": pl.String,
        "TUMOR_SIZE": pl.Int16,
    },
    "EHR_DIAGNOSES": {
        "DFCI_MRN": pl.String,
        "START_DT": pl.String,
        "END_DT": pl.String,
        # Up to three ICD-10 code/name pairs per diagnosis row. The name
        # columns feed COMPASS's compile_MRNs_for_manual_review.py, which
        # reports DIAGNOSIS_ICD10_NM alongside the code.
        "DIAGNOSIS_ICD10_CD": pl.String,
        "DIAGNOSIS_ICD10_NM": pl.String,
        "DIAGNOSIS_ICD10_CD2": pl.String,
        "DIAGNOSIS_ICD10_NM2": pl.String,
        "DIAGNOSIS_ICD10_CD3": pl.String,
        "DIAGNOSIS_ICD10_NM3": pl.String,
    },
    "GENOMIC_SPECIMEN": {
        "DFCI_MRN": pl.String,
        "UNIQUE_SAMPLE_ID": pl.Int64,
        "SAMPLE_ACCESSION_NBR": pl.String,
        "PRIMARY_CANCER_DIAGNOSIS": pl.String,
        "CANCER_TYPE": pl.String,
        "SAMPLE_COLLECTION_DT": pl.String,
        "REPORT_DT": pl.String,
        "TEST_ORDER_DT": pl.String,
        "MUTATIONAL_BURDEN": pl.Float64,
        "MISMATCH_REPAIR_STATUS": pl.String,
        "MICROSATELITEINSTABILITY_FMI_CD": pl.String,
        "TOBACCO_SIGNATURE": pl.String,
        "TEMOZOLOMIDE_SIGNATURE": pl.String,
        "POLE_SIGNATURE": pl.String,
        "APOBEC_SIGNATURE": pl.String,
        "UVA_SIGNATURE": pl.String,
        "TEST_TYPE": pl.String,
        "PANEL_VERSION": pl.String,
    },
    "CNV": {
        "DFCI_MRN": pl.String,
        "UNIQUE_SAMPLE_ID": pl.Int64,
        "SAMPLE_ACCESSION_NBR": pl.String,
        "GENE": pl.String,
        "CNV_TYPE_CD": pl.String,
        "COPY_COUNT": pl.Int16,
        "PANEL_VERSION": pl.String,
        "TEST_TYPE": pl.String,
    },
    "SNV": {
        "DFCI_MRN": pl.String,
        "UNIQUE_SAMPLE_ID": pl.Int64,
        "SAMPLE_ACCESSION_NBR": pl.String,
        "GENE": pl.String,
        "PATHOLOGIST_PATHOGENICITY": pl.String,
        "RAPIDHEME_KB_PATHOGENICITY": pl.String,
        "VARIANT_TYPE": pl.String,
        "HARMONIZED_HUGO_GENE_NAME": pl.String,
        "HARMONIZED_VARIANT_CLASS": pl.String,
        "TEST_TYPE": pl.String,
    },
    "SV": {
        "DFCI_MRN": pl.String,
        "UNIQUE_SAMPLE_ID": pl.Int64,
        "SAMPLE_ACCESSION_NBR": pl.String,
        "SV_TYPE": pl.String,
        "PARTNER1_HUGO_GENE_NM": pl.String,
        "PARTNER2_HUGO_GENE_NM": pl.String,
        "TIER": pl.String,
        "TEST_TYPE": pl.String,
    },
    "HEALTH_HISTORY": {
        "DFCI_MRN": pl.String,
        "START_DT": pl.String,
        "CODE": pl.String,
        "HEALTH_HISTORY_TYPE": pl.String,
        "CODE_TYPE": pl.String,
        "RESULTS": pl.String,
        "UNITS_CD": pl.String,
    },
    "MEDICATIONS": {
        "DFCI_MRN": pl.String,
        # NCI_PREFERRED_MED_NM is the spelling used by the OncDRS pulls and
        # by every downstream consumer. See ALIAS_MAP for releases that
        # spell it differently.
        "NCI_PREFERRED_MED_NM": pl.String,
        "MED_START_DT": pl.String,
    },
    "LABS": {
        "DFCI_MRN": pl.String,
        "SPECIMEN_COLLECT_DT": pl.String,
        "TEST_TYPE_CD": pl.String,
        "TEST_TYPE_DESCR": pl.String,
        "NUMERIC_RESULT": pl.Float64,
        "TEXT_RESULT": pl.String,
        "RESULT_UOM_NM": pl.String,
    },
    "PT_INFO_STATUS_REGISTRATION": {
        "DFCI_MRN": pl.String,
        "BIRTH_DT": pl.String,
        "CLIN_DEATH_DT": pl.String,
        "HYBRID_DEATH_DT": pl.String,
        "NDI_DEATH_DT": pl.String,
        # Censoring date. COMPASS reads DERIVED_LAST_ALIVE_DATE; see
        # ALIAS_MAP for releases that spell it DERIVED_LAST_CONTACT_DT.
        "DERIVED_LAST_ALIVE_DATE": pl.String,
        "GENDER_NM": pl.String,
        "GENDER_CD": pl.String,
        "PT_ONCOPANEL_PROFILED_IND": pl.String,
    },
    "TREATMENT_PLAN": {
        "DFCI_MRN": pl.String,
        "TREATMENT_PLAN_CATEGORY": pl.String,
        "STD_CHEMO_PLAN": pl.String,
        "RESEARCH_CHEMO_PLAN_NBR": pl.String,
        "OTHER_TREATMENT_PLAN": pl.String,
        "TPLAN_START_DT": pl.String,
    },
    "MEDICATIONS_SUMMARY": {
        "DFCI_MRN": pl.String,
        **{
            f"MED_ANTINEO_DRUG_CATEG_{i}": pl.String
            for i in range(1, 8)
        },
        **{
            f"MED_END_DT_{i}": pl.String
            for i in range(1, 8)
        },
        **{
            f"MED_LAST_ADMIN_DT_{i}": pl.String
            for i in range(1, 8)
        },
        **{
            f"MED_NCI_PREFERRED_NM_{i}": pl.String
            for i in range(1, 8)
        },
        **{
            f"MED_START_DT_{i}": pl.String
            for i in range(1, 8)
        },
    },
}

# Every clinical calendar field is normalized to pl.Date at ingestion. Source
# CSV columns are still read as strings so multiple release-specific formats can
# be handled explicitly before a strict final conversion.
DATE_COLUMN_MAP = {
    "CAREG": ["DIAGNOSIS_DT", "FIRST_CONTACT_DT", "LAST_CONTACT_DT"],
    "EHR_DIAGNOSES": ["START_DT", "END_DT"],
    "GENOMIC_SPECIMEN": [
        "SAMPLE_COLLECTION_DT",
        "REPORT_DT",
        "TEST_ORDER_DT",
    ],
    "HEALTH_HISTORY": ["START_DT"],
    "MEDICATIONS": ["MED_START_DT"],
    "LABS": ["SPECIMEN_COLLECT_DT"],
    "PT_INFO_STATUS_REGISTRATION": [
        "BIRTH_DT",
        "CLIN_DEATH_DT",
        "HYBRID_DEATH_DT",
        "NDI_DEATH_DT",
        "DERIVED_LAST_ALIVE_DATE",
    ],
    "TREATMENT_PLAN": ["TPLAN_START_DT"],
    "MEDICATIONS_SUMMARY": [
        *[f"MED_END_DT_{i}" for i in range(1, 8)],
        *[f"MED_LAST_ADMIN_DT_{i}" for i in range(1, 8)],
        *[f"MED_START_DT_{i}" for i in range(1, 8)],
    ],
}

_configured_date_fields = {
    (file_type, column)
    for file_type, columns in DATE_COLUMN_MAP.items()
    for column in columns
}
_date_like_schema_fields = {
    (file_type, column)
    for file_type, columns in COLUMN_MAP.items()
    for column in columns
    if column.endswith(("_DT", "_DATE")) or "_DT_" in column
}
if _configured_date_fields != _date_like_schema_fields:
    raise RuntimeError(
        "DATE_COLUMN_MAP coverage mismatch. "
        f"Missing: {sorted(_date_like_schema_fields - _configured_date_fields)}; "
        f"unexpected: {sorted(_configured_date_fields - _date_like_schema_fields)}"
    )

for _file_type, _date_columns in DATE_COLUMN_MAP.items():
    for _date_column in _date_columns:
        COLUMN_MAP[_file_type][_date_column] = pl.Date


def profile_date(column):
    """Normalize supported PROFILE date/timestamp strings, rejecting unknowns."""
    text = pl.col(column).cast(pl.String).str.strip_chars().replace("", None)

    iso_date = text.str.extract(r"^(\d{4}-\d{2}-\d{2})", 1)
    ymd_slash = (
        text
        .str.extract(r"^(\d{4}/\d{1,2}/\d{1,2})", 1)
        .str.strptime(pl.Date, format="%Y/%m/%d", strict=False)
        .dt.strftime("%Y-%m-%d")
    )
    mdy_slash = (
        text
        .str.extract(r"^(\d{1,2}/\d{1,2}/\d{4})", 1)
        .str.strptime(pl.Date, format="%m/%d/%Y", strict=False)
        .dt.strftime("%Y-%m-%d")
    )
    compact = (
        text
        .str.extract(r"^(\d{8})$", 1)
        .str.strptime(pl.Date, format="%Y%m%d", strict=False)
        .dt.strftime("%Y-%m-%d")
    )
    day_month_name = (
        text
        .str.extract(r"^(\d{1,2}-[A-Za-z]{3}-\d{4})", 1)
        .str.strptime(pl.Date, format="%d-%b-%Y", strict=False)
        .dt.strftime("%Y-%m-%d")
    )

    # The fallback deliberately carries unknown non-null text into a strict
    # parse, turning unexpected source formats into an actionable error rather
    # than silently converting them to null.
    normalized = pl.coalesce(
        iso_date,
        ymd_slash,
        mdy_slash,
        compact,
        day_month_name,
        pl.when(text.is_null()).then(None).otherwise(text),
    )
    return normalized.str.strptime(
        pl.Date, format="%Y-%m-%d", strict=True
    ).alias(column)


# Some releases spell the same field differently. Map each canonical
# COLUMN_MAP key to the alternate source spellings to try when the canonical
# name is absent from a release's header.
#
# Without this, a rename between releases silently yields an all-null column:
# the loop below inserts any unmatched requested column as null rather than
# failing. That is exactly what happened to NCI_PREFERRED_MED_NM (which would
# have wiped out every treatment-anchor and platinum computation downstream)
# and to DERIVED_LAST_ALIVE_DATE (which would have wiped out censoring).
#
# Aliases resolve to ONE canonical output column, so the output schema stays
# fixed and exact-full-row deduplication is not defeated by the same logical
# row arriving under two different column names in two different releases.
ALIAS_MAP = {
    "SNV": {
        "GENE": ["HARMONIZED_HUGO_GENE_NAME"],
    },
    "SV": {
        "PARTNER1_HUGO_GENE_NM": ["HARMONIZED_L_GENE"],
        "PARTNER2_HUGO_GENE_NM": ["HARMONIZED_R_GENE"],
    },
    "MEDICATIONS": {
        "NCI_PREFERRED_MED_NM": ["NCI_PREFERRED_MED_NAME"],
    },
    "PT_INFO_STATUS_REGISTRATION": {
        "DERIVED_LAST_ALIVE_DATE": ["DERIVED_LAST_CONTACT_DT"],
    },
}

SORT_COL_MAP = {
    'CAREG' : ['DFCI_MRN', 'DIAGNOSIS_DT'],
    'EHR_DIAGNOSES' : ['DFCI_MRN', 'START_DT'],
    'GENOMIC_SPECIMEN' : ['DFCI_MRN', 'SAMPLE_COLLECTION_DT'],
    'CNV' : ['DFCI_MRN', 'UNIQUE_SAMPLE_ID'],
    'SNV' : ['DFCI_MRN', 'UNIQUE_SAMPLE_ID'],
    'SV' : ['DFCI_MRN', 'UNIQUE_SAMPLE_ID'],
    'HEALTH_HISTORY' : ['DFCI_MRN', 'START_DT'],
    'MEDICATIONS' : ['DFCI_MRN', 'MED_START_DT'],
    'LABS' : ['DFCI_MRN', 'SPECIMEN_COLLECT_DT'],
    'PT_INFO_STATUS_REGISTRATION' : ['DFCI_MRN', 'DERIVED_LAST_ALIVE_DATE'],
    'TREATMENT_PLAN' : ['DFCI_MRN', 'TPLAN_START_DT'],
    'MEDICATIONS_SUMMARY' : ['DFCI_MRN', 'MED_START_DT_1']
}

null_values = [
    "null",
    "NULL",
    "None",
    "NA",
    "N/A",
    "",
    "Not Available from Source Panel"
]

OncDRS_PATH = Path('/data/gusev/PROFILE/CLINICAL/OncDRS/')
OUTPUT_PATH = Path('/data/gusev/USERS/jpconnor/data/PROFILE_DATA/')

def human_size(path: str | Path) -> str:
    size = Path(path).stat().st_size

    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size < 1024 or unit == "TB":
            return f"{size:.1f} {unit}"
        size /= 1024


import os
import polars as pl
from time import time
from pathlib import Path


OVERWRITE_EXISTING = False

# Restrict BOTH stages (per-release compression and the FINAL merge) to a
# subset of FILE_TYPES. None means all tables. Set this together with the
# OVERWRITE flags to rebuild only the tables whose COLUMN_MAP changed instead
# of re-reading every release of LABS.
REBUILD_FILE_TYPES = None

_ACTIVE_FILE_TYPES = [
    file_type
    for file_type in FILE_TYPES
    if REBUILD_FILE_TYPES is None or file_type in REBUILD_FILE_TYPES
]

for file_type in _ACTIVE_FILE_TYPES:
    file_type_schema = COLUMN_MAP[file_type]
    requested_cols = list(file_type_schema)
    sort_cols = SORT_COL_MAP[file_type]

    file_output_dir = OUTPUT_PATH / file_type
    file_output_dir.mkdir(parents=True, exist_ok=True)

    for data_pull_repo, data_pull_files in FILE_MAP.items():
        filename = data_pull_files.get(file_type)

        if filename is None:
            print(f"Skipping {data_pull_repo} {file_type}: no source file.")
            continue

        input_file = OncDRS_PATH / data_pull_repo / filename
        output_file = (
            file_output_dir
            / f"{data_pull_repo}_{file_type}.parquet"
        )
        temporary_file = output_file.with_suffix(".parquet.tmp")

        # Protect against accidental source/output overlap.
        input_resolved = input_file.resolve()
        output_resolved = output_file.resolve()

        if input_resolved == output_resolved:
            raise RuntimeError(
                f"Input and output paths are identical: {input_file}"
            )

        if not input_file.is_file():
            print(
                f"Skipping {data_pull_repo} {file_type}: "
                f"input does not exist: {input_file}"
            )
            continue

        if output_file.exists() and not OVERWRITE_EXISTING:
            existing_schema = pl.scan_parquet(output_file).collect_schema()
            expected_schema = pl.Schema(file_type_schema)
            if existing_schema == expected_schema:
                print(
                    f"Skipping {data_pull_repo} {file_type}: "
                    f"output already exists with the expected schema: {output_file}"
                )
                continue
            print(
                f"Rebuilding {data_pull_repo} {file_type}: existing output "
                "has a stale schema."
            )

        # Remove a leftover partial temporary file from a prior failed run.
        temporary_file.unlink(missing_ok=True)

        start_time = time()

        try:
            # Inspect only the header/schema before processing the full file.
            source_schema = pl.scan_csv(
                input_file,
                infer_schema_length=0,
            ).collect_schema()

            # Resolve each canonical column to the source column that
            # actually carries it: itself when present, otherwise the first
            # matching ALIAS_MAP spelling, otherwise None (null-filled).
            aliases = ALIAS_MAP.get(file_type, {})
            resolved = {}
            for col in requested_cols:
                if col in source_schema:
                    resolved[col] = col
                    continue
                resolved[col] = next(
                    (
                        alias
                        for alias in aliases.get(col, ())
                        if alias in source_schema
                    ),
                    None,
                )

            available_cols = [
                col
                for col in requested_cols
                if resolved[col] is not None
            ]

            missing_cols = [
                col
                for col in requested_cols
                if resolved[col] is None
            ]

            aliased_cols = {
                col: resolved[col]
                for col in available_cols
                if resolved[col] != col
            }

            if aliased_cols:
                print(
                    f"{data_pull_repo} {file_type}: "
                    f"resolved via alias: {aliased_cols}"
                )

            if missing_cols:
                print(
                    f"{data_pull_repo} {file_type}: "
                    f"missing columns will be inserted as null: "
                    f"{missing_cols}"
                )

            date_columns = set(DATE_COLUMN_MAP.get(file_type, ()))
            available_schema = {
                resolved[col]: (
                    pl.String if col in date_columns else file_type_schema[col]
                )
                for col in available_cols
            }

            cur_file = (
                pl.scan_csv(
                    input_file,
                    schema_overrides=available_schema,
                    try_parse_dates=False,
                    ignore_errors=False,
                    null_values=null_values,
                )
                .select(
                    [
                        (
                            pl.col(resolved[col]).alias(col)
                            if resolved[col] is not None
                            else pl.lit(None, dtype=dtype).alias(col)
                        )
                        for col, dtype in file_type_schema.items()
                    ]
                )
                .with_columns(
                    [
                        profile_date(col)
                        for col in date_columns
                        if resolved[col] is not None
                    ]
                )
            )

            if file_type in SOMATIC_RESULT_FILE_TYPES:
                specimen_file = (
                    OUTPUT_PATH
                    / "GENOMIC_SPECIMEN"
                    / f"{data_pull_repo}_GENOMIC_SPECIMEN.parquet"
                )
                if not specimen_file.is_file():
                    raise RuntimeError(
                        f"{file_type}: missing same-release specimen lookup: "
                        f"{specimen_file}"
                    )
                specimen_test_type = (
                    pl.scan_parquet(specimen_file)
                    .select(
                        "DFCI_MRN",
                        "UNIQUE_SAMPLE_ID",
                        pl.col("TEST_TYPE").alias("_SPECIMEN_TEST_TYPE"),
                    )
                    .unique(
                        subset=["DFCI_MRN", "UNIQUE_SAMPLE_ID"],
                        keep="first",
                    )
                )
                cur_file = (
                    cur_file
                    .join(
                        specimen_test_type,
                        on=["DFCI_MRN", "UNIQUE_SAMPLE_ID"],
                        how="left",
                    )
                    .with_columns(
                        pl.coalesce("_SPECIMEN_TEST_TYPE", "TEST_TYPE")
                        .alias("TEST_TYPE")
                    )
                    .drop("_SPECIMEN_TEST_TYPE")
                )

            if file_type in ({"GENOMIC_SPECIMEN"} | SOMATIC_RESULT_FILE_TYPES):
                null_group_rows = (
                    cur_file
                    .filter(
                        pl.any_horizontal(
                            [pl.col(col).is_null() for col in SOMATIC_GROUP_KEY]
                        )
                    )
                    .select(pl.len().alias("n"))
                    .collect(engine="streaming")
                    .item()
                )
                if null_group_rows:
                    raise RuntimeError(
                        f"{data_pull_repo} {file_type}: {null_group_rows:,} rows "
                        f"have null testing-group keys: {SOMATIC_GROUP_KEY}"
                    )

            available_sort_cols = [
                col for col in sort_cols
                if col in available_cols
            ]

            if available_sort_cols:
                cur_file = cur_file.sort(
                    available_sort_cols,
                    descending=True,
                )

            # Write to a temporary path first.
            cur_file.sink_parquet(
                temporary_file,
                compression="zstd",
                compression_level=3,
                statistics=True,
            )

            # Basic validation before replacing/creating the final output.
            output_schema = pl.scan_parquet(
                temporary_file
            ).collect_schema()

            expected_output_schema = pl.Schema(file_type_schema)
            if output_schema != expected_output_schema:
                raise RuntimeError(
                    f"Generated Parquet schema does not match {file_type}.\n"
                    f"Expected: {expected_output_schema}\n"
                    f"Found:    {output_schema}"
                )

            # Atomic replacement on the same filesystem.
            os.replace(temporary_file, output_file)

        except Exception:
            temporary_file.unlink(missing_ok=True)
            raise

        elapsed_minutes = (time() - start_time) / 60

        print(f"Compressing {data_pull_repo} {file_type} complete.")
        print(f"Time elapsed = {elapsed_minutes:.2f} minutes")
        print(f"Previous file size = {human_size(input_file)}")
        print(f"New file size = {human_size(output_file)}\n")


# ===================================================================
# MERGE RELEASE PARQUETS INTO ONE DEDUPLICATED FILE PER TABLE
# ===================================================================

import os
import polars as pl
from pathlib import Path
from time import time


# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------

FINAL_OUTPUT_DIR = OUTPUT_PATH / "FINAL"
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OVERWRITE_FINAL = False

# Explicit chronological order of releases.
# Later releases receive larger values and are retained when the same
# logical record appears in more than one release.
RELEASE_ORDER = {
    release: rank
    for rank, release in enumerate(FILE_MAP.keys())
}


# -------------------------------------------------------------------
# Conservative deduplication keys
# -------------------------------------------------------------------
#
# A value of None means:
#     Remove only rows that are identical across every data column.
#
# A list of columns means:
#     Treat those columns as the logical record identifier and keep
#     the version from the newest release.
#
# Tables without a sufficiently reliable record identifier use exact
# full-row deduplication to minimize accidental data loss.
# -------------------------------------------------------------------

DEDUP_COL_MAP = {
    # Registry diagnosis identifier is reasonably strong within patient.
    "CAREG": [
        "DFCI_MRN",
        "DIAGNOSIS_SEQ_NBR",
        "TUMOR_OCC_ORD",
    ],

    # No dedicated diagnosis-row identifier is available.
    # Include all main identifying diagnosis fields.
    "EHR_DIAGNOSES": [
        "DFCI_MRN",
        "START_DT",
        "END_DT",
        "DIAGNOSIS_ICD10_CD",
        "DIAGNOSIS_ICD10_CD2",
        "DIAGNOSIS_ICD10_CD3",
    ],

    # SAMPLE_ACCESSION_NBR is an accessioned testing event and may contain
    # multiple local sample IDs. TEST_TYPE preserves distinct DNA/RNA or other
    # assay groupings while allowing release-specific UNIQUE_SAMPLE_ID values
    # to collapse into one stable cross-release testing group.
    "GENOMIC_SPECIMEN": [
        "DFCI_MRN",
        "SAMPLE_ACCESSION_NBR",
        "TEST_TYPE",
    ],

    # Somatic calls use the same stable testing-group key. UNIQUE_SAMPLE_ID is
    # deliberately excluded because it changes between data requests.
    "CNV": [
        "DFCI_MRN",
        "SAMPLE_ACCESSION_NBR",
        "TEST_TYPE",
        "GENE",
        "CNV_TYPE_CD",
    ],

    # Use every available call descriptor except release-local sample ID, so
    # distinct variants remain distinct while repeated releases collapse.
    "SNV": [
        "DFCI_MRN",
        "SAMPLE_ACCESSION_NBR",
        "TEST_TYPE",
        "GENE",
        "PATHOLOGIST_PATHOGENICITY",
        "RAPIDHEME_KB_PATHOGENICITY",
        "VARIANT_TYPE",
        "HARMONIZED_HUGO_GENE_NAME",
        "HARMONIZED_VARIANT_CLASS",
    ],

    "SV": [
        "DFCI_MRN",
        "SAMPLE_ACCESSION_NBR",
        "TEST_TYPE",
        "SV_TYPE",
        "PARTNER1_HUGO_GENE_NM",
        "PARTNER2_HUGO_GENE_NM",
        "TIER",
    ],

    # No reliable source-row identifier.
    "HEALTH_HISTORY": None,

    # Repeated medication orders can legitimately share drug and date.
    "MEDICATIONS": None,

    # Wide summary records may change between releases.
    # Keep the newest summary row for each patient.
    "MEDICATIONS_SUMMARY": [
        "DFCI_MRN",
    ],

    # Multiple legitimate lab results can share patient, time, and value.
    # Exact-row deduplication avoids collapsing repeated measurements.
    "LABS": None,

    # Patient-level status table. See COALESCE_FILE_TYPES below: this is
    # handled by a per-column coalesce instead of a row-key dedup, so this
    # entry is only used for the missing-dedup-column existence check.
    "PT_INFO_STATUS_REGISTRATION": [
        "DFCI_MRN",
    ],

    # Treatment plans can have duplicate-looking but distinct records.
    "TREATMENT_PLAN": None,
}


# -------------------------------------------------------------------
# Per-column coalesce (instead of per-row newest-wins dedup)
# -------------------------------------------------------------------
#
# GENOMIC_SPECIMEN and PT_INFO_STATUS_REGISTRATION are entity-level tables
# whose newer releases may omit values retained in older releases. For genomic
# specimens, a null-filled date in a newer release must not replace a populated
# SAMPLE_COLLECTION_DT, TEST_ORDER_DT, or REPORT_DT from an older release.
# Similarly, ALL_2026_03 maps patient status to IDM_D_PATIENT_STATUS.csv -- a
# visit-tracking table that structurally lacks BIRTH_DT, GENDER_NM, and
# the death/censoring columns (they arrive null-filled by the Stage 1
# missing-column logic above). Under a per-row newest-wins dedup, that
# release's all-null row for a patient replaces perfectly good data from
# an older release for ~124.5K of ~125K patients: BIRTH_DT non-null drops
# from 100% to ~6.7% overall once ALL_2026_03 is folded in.
#
# Coalescing per column instead of per row fixes both cases: for each logical
# key, each output column takes the value from the newest release in which THAT
# COLUMN is non-null. A column ALL_2026_03 does populate still wins, since it is
# the newest source consulted first.
#
# These tables are small compared with row-exploded LABS, MEDICATIONS, and
# EHR_DIAGNOSES, so the coalesce is done
# with a single grouped aggregation rather than extending the streaming
# anti-join fold used for the other, much larger tables -- that fold's
# memory-avoidance rationale doesn't apply at this row count, and
# per-column coalesce isn't expressible as a row-level anti-join anyway.
COALESCE_FILE_TYPES = {"GENOMIC_SPECIMEN", "PT_INFO_STATUS_REGISTRATION"}


# -------------------------------------------------------------------
# Merge and deduplicate
# -------------------------------------------------------------------
#
# Releases are folded in one at a time, newest first, instead of
# concatenating every release's Parquet file and deduplicating the
# whole pile in one step.
#
# Each release's surviving rows are written to their own file exactly
# once and are never rewritten afterwards. To decide what survives, a
# release is anti-joined against the key columns of the files already
# written, read lazily -- projection pushdown means only the key
# columns are actually scanned. Nothing is accumulated or re-emitted
# between steps, so total write volume is one pass over the kept rows
# rather than one pass per release.
#
# Entity-level tables in COALESCE_FILE_TYPES instead take a per-column
# coalesce branch below, since a per-row anti-join fold cannot express
# "keep this entity's row but prefer an older release's value in this
# one column."
# -------------------------------------------------------------------

ordered_releases = sorted(
    RELEASE_ORDER,
    key=RELEASE_ORDER.get,
    reverse=True,
)

for file_type in _ACTIVE_FILE_TYPES:
    start_time = time()

    file_output_dir = OUTPUT_PATH / file_type
    output_file = FINAL_OUTPUT_DIR / f"{file_type}.parquet"
    temporary_file = FINAL_OUTPUT_DIR / f".{file_type}.parquet.tmp"

    if output_file.exists() and not OVERWRITE_FINAL:
        existing_lf = pl.scan_parquet(output_file)
        existing_schema = existing_lf.collect_schema()
        expected_schema = pl.Schema(COLUMN_MAP[file_type])
        grouping_is_current = True
        if file_type == "GENOMIC_SPECIMEN" and existing_schema == expected_schema:
            grouping_is_current = (
                existing_lf
                .group_by(SOMATIC_GROUP_KEY)
                .len()
                .filter(pl.col("len") > 1)
                .limit(1)
                .collect(engine="streaming")
                .is_empty()
            )
        if existing_schema == expected_schema and grouping_is_current:
            print(
                f"Skipping {file_type}: final output already exists "
                f"with the expected schema and grouping:\n  {output_file}\n"
            )
            continue
        print(
            f"Rebuilding {file_type}: final output has a stale schema "
            "or grouping grain."
        )

    data_columns = list(COLUMN_MAP[file_type])
    dedup_cols = DEDUP_COL_MAP[file_type]

    if dedup_cols is not None:
        missing_dedup_cols = [
            col for col in dedup_cols if col not in data_columns
        ]

        if missing_dedup_cols:
            raise RuntimeError(
                f"{file_type}: deduplication columns are missing: "
                f"{missing_dedup_cols}"
            )

        key_cols = dedup_cols
        dedup_description = (
            "newest-release logical deduplication using "
            f"{dedup_cols}"
        )
    else:
        key_cols = data_columns
        dedup_description = "exact full-row deduplication"

    if file_type in COALESCE_FILE_TYPES:
        dedup_description = (
            f"per-column coalesce across releases keyed on {key_cols}"
        )

    temporary_file.unlink(missing_ok=True)
    # Clear any per-release "kept" files left over from a prior failed run.
    for stale_file in FINAL_OUTPUT_DIR.glob(
        f".{file_type}.kept.*.parquet.tmp"
    ):
        stale_file.unlink(missing_ok=True)

    source_files = [
        file_output_dir / f"{release}_{file_type}.parquet"
        for release in ordered_releases
        if (file_output_dir / f"{release}_{file_type}.parquet").is_file()
    ]

    kept_files = []
    rows_before = 0

    try:
        if file_type in COALESCE_FILE_TYPES:
            # ---------------------------------------------------------
            # Per-column coalesce branch.
            #
            # Tag each release's rows with its RELEASE_ORDER rank, stack
            # every release, then group by the key columns and take, for
            # each data column independently, the value from the row
            # with the highest rank where that column is non-null.
            # pl.col(...).drop_nulls().first() on a rank-descending-
            # sorted-within-group column achieves exactly that: nulls
            # are dropped before "first" is taken, so a newer release's
            # null in one column falls through to an older release's
            # non-null value in that same column, while every column
            # still independently prefers the newest release that has
            # data for it.
            # ---------------------------------------------------------
            if not source_files:
                print(
                    f"Skipping {file_type}: "
                    f"no release-specific Parquet files found.\n"
                )
                continue

            release_frames = []
            for source_file in source_files:
                release = source_file.name[: -len(f"_{file_type}.parquet")]
                release_lf = pl.scan_parquet(source_file).with_columns(
                    pl.lit(RELEASE_ORDER[release]).alias("_release_rank")
                )
                release_rows = (
                    release_lf
                    .select(pl.len().alias("n"))
                    .collect(engine="streaming")
                    .item()
                )
                rows_before += release_rows
                release_frames.append(release_lf)

            stacked_lf = pl.concat(release_frames, how="vertical")

            non_key_cols = [
                col for col in data_columns if col not in key_cols
            ]

            if file_type == "GENOMIC_SPECIMEN":
                # Temporal rules for a grouped accession/test: the earliest
                # order and collection bound the testing event, while the
                # latest report is the conservative availability date.
                specimen_date_aggs = {
                    "SAMPLE_COLLECTION_DT": pl.col("SAMPLE_COLLECTION_DT").min(),
                    "TEST_ORDER_DT": pl.col("TEST_ORDER_DT").min(),
                    "REPORT_DT": pl.col("REPORT_DT").max(),
                }
            else:
                specimen_date_aggs = {}

            deduplicated_lf = (
                stacked_lf
                .sort("_release_rank", descending=True)
                .group_by(key_cols, maintain_order=False)
                .agg(
                    [
                        specimen_date_aggs.get(
                            col, pl.col(col).drop_nulls().first()
                        ).alias(col)
                        for col in non_key_cols
                    ]
                )
                .select(data_columns)
            )
        else:
            for release in ordered_releases:
                input_file = (
                    file_output_dir
                    / f"{release}_{file_type}.parquet"
                )

                if not input_file.is_file():
                    continue

                release_lf = pl.scan_parquet(input_file)

                release_rows = (
                    release_lf
                    .select(pl.len().alias("n"))
                    .collect(engine="streaming")
                    .item()
                )
                rows_before += release_rows

                if not kept_files:
                    # First (newest) release: only intra-file duplicates
                    # need to be collapsed.
                    survivors_lf = release_lf.unique(
                        subset=key_cols,
                        keep="first",
                        maintain_order=False,
                    )
                else:
                    # Keys already contributed by a newer release. These
                    # files are only ever read, never rewritten, and
                    # projection pushdown limits the scan to key_cols.
                    seen_keys_lf = pl.concat(
                        [
                            pl.scan_parquet(kept_file).select(key_cols)
                            for kept_file in kept_files
                        ],
                        how="vertical",
                    )

                    # Drop rows whose key was already kept, then collapse
                    # duplicate keys within this release itself.
                    survivors_lf = (
                        release_lf
                        .join(
                            seen_keys_lf,
                            on=key_cols,
                            how="anti",
                            # Match .unique()'s null-equals-null semantics;
                            # polars joins otherwise treat null != null,
                            # which would let null-keyed duplicates through.
                            nulls_equal=True,
                        )
                        .unique(
                            subset=key_cols,
                            keep="first",
                            maintain_order=False,
                        )
                    )

                # Persist this release's surviving rows exactly once.
                kept_file = (
                    FINAL_OUTPUT_DIR
                    / f".{file_type}.kept.{release}.parquet.tmp"
                )
                survivors_lf.sink_parquet(
                    kept_file,
                    compression="zstd",
                    compression_level=3,
                    # Statistics on a temporary file nothing ever queries
                    # directly are pure write overhead; only the final
                    # output benefits from them.
                    statistics=False,
                )
                kept_files.append(kept_file)

            if not kept_files:
                print(
                    f"Skipping {file_type}: "
                    f"no release-specific Parquet files found.\n"
                )
                continue

            # Combine every release's kept rows. Each file is read exactly
            # once here; none of them were ever rewritten during the loop.
            deduplicated_lf = pl.concat(
                [pl.scan_parquet(kept_file) for kept_file in kept_files],
                how="vertical",
            )

        # Apply the desired final sort order.
        available_sort_cols = [
            col
            for col in SORT_COL_MAP[file_type]
            if col in data_columns
        ]

        if available_sort_cols:
            deduplicated_lf = deduplicated_lf.sort(
                available_sort_cols,
                descending=True,
            )

        # Count final rows before writing.
        rows_after = (
            deduplicated_lf
            .select(pl.len().alias("n"))
            .collect(engine="streaming")
            .item()
        )

        rows_removed = rows_before - rows_after

        # Write to a temporary file first.
        #
        # Parquet compresses each page (~1 MB) independently, so zstd's
        # higher levels have very little window to work with: measured
        # on this data, level 19 cost ~7x the write time for ~3% less
        # size, while level 9 gets most of that for ~1.3x.
        deduplicated_lf.sink_parquet(
            temporary_file,
            compression="zstd",
            compression_level=9,
            statistics=True,
        )

        # -----------------------------------------------------------
        # Validate temporary output
        # -----------------------------------------------------------

        validation_lf = pl.scan_parquet(temporary_file)
        output_schema = validation_lf.collect_schema()

        expected_output_schema = pl.Schema(COLUMN_MAP[file_type])
        if output_schema != expected_output_schema:
            raise RuntimeError(
                f"{file_type}: unexpected output schema.\n"
                f"Expected: {expected_output_schema}\n"
                f"Found:    {output_schema}"
            )

        validated_rows = (
            validation_lf
            .select(pl.len().alias("n"))
            .collect(engine="streaming")
            .item()
        )

        if validated_rows != rows_after:
            raise RuntimeError(
                f"{file_type}: output row-count validation failed. "
                f"Expected {rows_after:,}, found {validated_rows:,}."
            )

        # Atomically move the validated temporary file into place.
        os.replace(temporary_file, output_file)

    except Exception:
        temporary_file.unlink(missing_ok=True)
        for kept_file in kept_files:
            kept_file.unlink(missing_ok=True)
        raise

    # Clean up the per-release temporary files on success too.
    for kept_file in kept_files:
        kept_file.unlink(missing_ok=True)

    elapsed_minutes = (time() - start_time) / 60

    print(f"Completed {file_type}")
    print(f"  Strategy:      {dedup_description}")
    print(f"  Input files:   {len(source_files)}")
    print(f"  Rows before:   {rows_before:,}")
    print(f"  Rows after:    {rows_after:,}")
    print(f"  Rows removed:  {rows_removed:,}")
    print(f"  Output size:   {human_size(output_file)}")
    print(f"  Time elapsed:  {elapsed_minutes:.2f} minutes")
    print(f"  Output:        {output_file}\n")

In [ ]:
# Compare the combined on-disk size of all source CSVs with each final Parquet.
def format_bytes(size_bytes):
    size = float(size_bytes)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size < 1024 or unit == "TB":
            return f"{size:.1f} {unit}"
        size /= 1024


print("Source CSV rows/size compared with final compressed Parquet")
print("-" * 153)
print(
    f"{'File type':<31} {'CSVs':>4} {'Source rows':>15} "
    f"{'Final rows':>15} {'Rows removed':>15} "
    f"{'Source total':>14} {'Final Parquet':>14} "
    f"{'Reduction':>11} {'Ratio':>9}"
)
print("-" * 153)

grand_source_bytes = 0
grand_final_bytes = 0
grand_source_count = 0
grand_source_rows = 0
grand_final_rows = 0
all_final_files_present = True

for file_type in FILE_TYPES:
    source_files = [
        OncDRS_PATH / release / filename
        for release, release_files in FILE_MAP.items()
        if (filename := release_files.get(file_type)) is not None
        and (OncDRS_PATH / release / filename).is_file()
    ]
    final_file = FINAL_OUTPUT_DIR / f"{file_type}.parquet"

    source_bytes = sum(path.stat().st_size for path in source_files)
    final_bytes = final_file.stat().st_size if final_file.is_file() else 0
    source_rows = sum(
        pl.scan_csv(path, infer_schema_length=0)
        .select(pl.len().alias("n"))
        .collect(engine="streaming")
        .item()
        for path in source_files
    )
    final_rows = (
        pl.scan_parquet(final_file)
        .select(pl.len().alias("n"))
        .collect(engine="streaming")
        .item()
        if final_file.is_file()
        else None
    )

    grand_source_bytes += source_bytes
    grand_final_bytes += final_bytes
    grand_source_count += len(source_files)
    grand_source_rows += source_rows
    if final_rows is None:
        all_final_files_present = False
    else:
        grand_final_rows += final_rows

    if source_bytes and final_bytes:
        reduction = 100 * (1 - final_bytes / source_bytes)
        ratio = source_bytes / final_bytes
        reduction_text = f"{reduction:.1f}%"
        ratio_text = f"{ratio:.1f}x"
    else:
        reduction_text = "n/a"
        ratio_text = "n/a"

    final_text = format_bytes(final_bytes) if final_bytes else "missing"
    final_rows_text = f"{final_rows:,}" if final_rows is not None else "missing"
    rows_removed_text = (
        f"{source_rows - final_rows:,}"
        if final_rows is not None
        else "n/a"
    )
    print(
        f"{file_type:<31} {len(source_files):>4} "
        f"{source_rows:>15,} {final_rows_text:>15} "
        f"{rows_removed_text:>15} "
        f"{format_bytes(source_bytes):>14} {final_text:>14} "
        f"{reduction_text:>11} {ratio_text:>9}"
    )

print("-" * 153)
if grand_source_bytes and grand_final_bytes:
    grand_reduction = 100 * (1 - grand_final_bytes / grand_source_bytes)
    grand_ratio = grand_source_bytes / grand_final_bytes
    grand_reduction_text = f"{grand_reduction:.1f}%"
    grand_ratio_text = f"{grand_ratio:.1f}x"
else:
    grand_reduction_text = "n/a"
    grand_ratio_text = "n/a"

grand_final_rows_text = (
    f"{grand_final_rows:,}"
    if all_final_files_present
    else "partial"
)
grand_rows_removed_text = (
    f"{grand_source_rows - grand_final_rows:,}"
    if all_final_files_present
    else "n/a"
)

print(
    f"{'TOTAL':<31} {grand_source_count:>4} "
    f"{grand_source_rows:>15,} {grand_final_rows_text:>15} "
    f"{grand_rows_removed_text:>15} "
    f"{format_bytes(grand_source_bytes):>14} "
    f"{format_bytes(grand_final_bytes):>14} "
    f"{grand_reduction_text:>11} {grand_ratio_text:>9}"
)
